# 🎙️ AIC 2026 - Video/Audio ASR Extraction Pipeline (faster-whisper)
### 🎯 Kaggle GPU Edition - Trích xuất giọng nói cho video từ **L21 đến L30**

Notebook này sử dụng mô hình **faster-whisper (Large-v3-Turbo)** để trích xuất giọng nói tiếng Việt:
- Nhanh gấp 4–5 lần Whisper nguyên bản nhờ CTranslate2 FP16 trên GPU T4.
- Tự động quét cực nhanh các file video/audio (.mp4, .mkv, .mp3, .wav, .m4a).
- Tự động căn chỉnh timestamp và map với keyframe gần nhất (`nearest_faiss_id`, `nearest_frame_name`).
- Xuất file **`asr_results.json`** đúng 100% schema Elasticsearch `aic_asr` của Backend.

---

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q faster-whisper transformers torch torchaudio tqdm Pillow
!apt-get update -qq && apt-get install -y -qq ffmpeg

## ⚙️ 2. Cấu hình & Tự động Phát hiện Thư mục Videos/Media trong Input

In [ ]:
import os
import sys
import glob
import json
import csv
import re
import warnings
from pathlib import Path
from tqdm import tqdm
import torch

warnings.filterwarnings("ignore")

OUTPUT_JSON = Path("/kaggle/working/asr_results.json")
CHECKPOINT_JSON = Path("/kaggle/working/asr_results_checkpoint.json")

# Tự động tìm thư mục Videos/Media trong /kaggle/input/
MEDIA_DIR = None
search_base = "/kaggle/input"

if os.path.exists(search_base):
    for root, dirs, files in os.walk(search_base):
        # Tìm thư mục có chứa file video
        has_video = any(f.lower().endswith(('.mp4', '.mkv', '.avi', '.mp3', '.wav', '.m4a')) for f in files[:20])
        if has_video:
            MEDIA_DIR = Path(root)
            break

if not MEDIA_DIR:
    if os.path.exists("D:/AIC_Data/Videos"):
        MEDIA_DIR = Path("D:/AIC_Data/Videos")
    else:
        MEDIA_DIR = Path("/kaggle/input/your-videos-dataset/")

print(f"📁 Thư mục Videos/Media tìm thấy: {MEDIA_DIR}")
print(f"💾 File kết quả đầu ra: {OUTPUT_JSON}")

## 🔗 3. Xây dựng Keyframe Timestamp Index để Alignment

In [ ]:
def build_keyframe_index():
    video_keyframe_map = {}
    faiss_id_counter = 0
    
    # Quét CSVs trong /kaggle/input/ (chỉ quét các file kết thúc bằng .csv)
    csv_files = []
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.lower().endswith('.csv'):
                    csv_files.append(os.path.join(root, f))
                    
    for csv_f in csv_files:
        v_id = os.path.splitext(os.path.basename(csv_f))[0]
        # Chỉ xử lý các file CSV video L21 đến L30
        if not re.search(r"L(2[1-9]|30)_V\d+", v_id):
            continue
            
        try:
            with open(csv_f, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                frames_list = []
                for row in reader:
                    try:
                        n_val = int(row.get('n', 0))
                        pts_time = float(row.get('pts_time', 0.0))
                        frames_list.append({
                            "faiss_id": faiss_id_counter,
                            "timestamp": pts_time,
                            "frame_name": f"{n_val:04d}.webp"
                        })
                        faiss_id_counter += 1
                    except Exception:
                        pass
                if frames_list:
                    video_keyframe_map[v_id] = sorted(frames_list, key=lambda x: x["timestamp"])
        except Exception:
            pass
            
    if video_keyframe_map:
        print(f"✅ Đã nạp keyframes map cho {len(video_keyframe_map)} videos từ CSVs.")
    else:
        print("ℹ️ Chưa có map-keyframes CSVs. Hệ thống sẽ căn chỉnh frame ước lượng theo giây.")
        
    return video_keyframe_map

def find_nearest_keyframe(target_time, video_id, keyframe_index):
    if video_id not in keyframe_index or not keyframe_index[video_id]:
        return {"faiss_id": 0, "frame_name": f"{int(target_time):04d}.webp"}
    frames = keyframe_index[video_id]
    return min(frames, key=lambda f: abs(f["timestamp"] - target_time))

keyframe_index = build_keyframe_index()

## 🤖 4. Khởi tạo Whisper Model trên GPU (`large-v3-turbo`)

In [ ]:
from faster_whisper import WhisperModel

# Dùng model large-v3-turbo (chính xác cao nhất cho tiếng Việt và tốc độ cực nhanh trên T4)
MODEL_SIZE = "large-v3-turbo"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"

print(f"Đang nạp model faster-whisper [{MODEL_SIZE}] trên GPU ({COMPUTE_TYPE})...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("✅ Whisper model đã sẵn sàng trên GPU!")

## ⚡ 5. Thực hiện Trích xuất ASR & Căn chỉnh Keyframe (L21 -> L30)

In [ ]:
valid_video_exts = {'.mp4', '.mkv', '.avi', '.mov', '.webm', '.mp3', '.wav', '.m4a', '.flac'}
all_media = []

if MEDIA_DIR and os.path.exists(MEDIA_DIR):
    for root, _, files in os.walk(MEDIA_DIR):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext in valid_video_exts:
                all_media.append(os.path.join(root, fname))

# Lọc video từ L21 đến L30
media_files = []
for mf in all_media:
    basename = os.path.basename(mf)
    if re.search(r"L(2[1-9]|30)_V\d+", basename):
        media_files.append(mf)

if not media_files:
    media_files = sorted(all_media) # fallback nếu tên không chứa tiền tố Lxx
else:
    media_files = sorted(media_files)
    
print(f"🔥 Tìm thấy tổng cộng {len(media_files)} video/audio thuộc dải L21 -> L30 để xử lý.")

# Nạp checkpoint đã xử lý trước đó
processed_videos = set()
asr_results = []
if CHECKPOINT_JSON.exists():
    try:
        with open(CHECKPOINT_JSON, 'r', encoding='utf-8') as f:
            asr_results = json.load(f)
            processed_videos = {item["video_id"] for item in asr_results}
        print(f"🔄 Đã nạp {len(asr_results)} đoạn ASR từ checkpoint ({len(processed_videos)} videos).")
    except Exception as e:
        print("Lỗi nạp checkpoint:", e)

for media_path in tqdm(media_files, desc="🚀 GPU Trích xuất ASR L21 -> L30"):
    video_id = os.path.splitext(os.path.basename(media_path))[0]
    # Lấy định danh video dạng L21_V001
    m = re.search(r"(L\d+_V\d+)", video_id)
    if m:
        video_id = m.group(1)
        
    if video_id in processed_videos:
        continue
        
    try:
        # faster-whisper đọc trực tiếp video .mp4 qua ffmpeg C-API
        segments, info = model.transcribe(
            media_path,
            language="vi",
            beam_size=5,
            word_timestamps=False,
            vad_filter=True, # Bỏ qua đoạn im lặng
            vad_parameters=dict(min_silence_duration_ms=500)
        )
        
        video_segments = []
        for seg in segments:
            start_time = round(float(seg.start), 3)
            end_time = round(float(seg.end), 3)
            text = seg.text.strip()
            if not text:
                continue
                
            mid_time = (start_time + end_time) / 2.0
            nearest_frame = find_nearest_keyframe(mid_time, video_id, keyframe_index)
            
            doc = {
                "video_id": video_id,
                "start_time": start_time,
                "end_time": end_time,
                "text": text,
                "nearest_faiss_id": int(nearest_frame["faiss_id"]),
                "nearest_frame_name": str(nearest_frame["frame_name"])
            }
            video_segments.append(doc)
            
        asr_results.extend(video_segments)
        processed_videos.add(video_id)
        
        # Lưu checkpoint sau mỗi video
        os.makedirs(os.path.dirname(CHECKPOINT_JSON), exist_ok=True)
        with open(CHECKPOINT_JSON, 'w', encoding='utf-8') as f:
            json.dump(asr_results, f, ensure_ascii=False, indent=2)
            
    except Exception as e:
        print(f"Lỗi xử lý {media_path}: {e}")

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(asr_results, f, ensure_ascii=False, indent=2)

print(f"\n🎉 HOÀN TẤT TOÀN BỘ! Đã lưu {len(asr_results)} đoạn ASR vào: {OUTPUT_JSON}")

## 📊 6. Kiểm tra & Hướng dẫn Tải về

In [ ]:
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Tổng số câu ASR trích xuất: {len(data)}")
    if data:
        print("\n--- 3 MẪU ĐẦU TIÊN ---")
        print(json.dumps(data[:3], indent=2, ensure_ascii=False))
    print(f"\n👉 File kết quả đã sẵn sàng tại: {OUTPUT_JSON}")
    print("1. Ở cột bên phải Kaggle (tab Output), nhấn vào file `asr_results.json` để tải về máy.")
    print("2. Copy file vào thư mục Backend: `src/dict/asr_results.json`.")
    print("3. Chạy `python scripts/indexing/master_index_pipeline.py` để nạp vào Elasticsearch.")